# STANCEOSAURUS BENCHMARK: SITPCL WITH BERTWEET BACKBONE (FAIR BENCHMARK)
Notebook này hiện thực giải pháp **SITPCL** (*Speaker Interaction and Target-aware Prototypical Contrastive Learning* - ACL 2025 Findings) trên tập dữ liệu hội thoại đa bên đã chuẩn hóa **Stanceosaurus Multi-party English** (`train_conversations.json`, `dev_conversations.json`, `test_conversations.json`).

### ⚖️ NGUYÊN TẮC ĐỐI SOÁT CÔNG BẰNG (FAIR PROTOCOL):
1. **Cùng Backbone:** Sử dụng cùng backbone `vinai/bertweet-base` với mô hình đối thủ Baseline.
2. **Cùng Dữ Liệu & Phân Chia Unseen Claims:** Dùng chung 100% tập dữ liệu hội thoại đa bên đã cân bằng độ sâu ($K=6$).
3. **Cùng Effective Batch Size:** Batch size 16 với tích lũy gradient 2 bước = 32.
4. **Cùng Hàm Mất Mát Cơ Sở:** Áp dụng Class-Balanced Focal Loss ($\gamma = 2.0$, cùng trọng số lớp) kết hợp cùng Target Prototypical Contrastive Loss ($L_{total} = L_{Focal} + \alpha L_{TPC}$).
5. **Kích hoạt trọn vẹn SIE:** Tận dụng 48.4% mẫu có cùng người nói lặp lại để học *Intra-speaker Attention* và hơn 41% mẫu có $\ge 3$ người để học *Inter-speaker Attention*.

### 💡 CƠ CHẾ CHẠY TEST NHANH VÀ TREO MÁY QUA ĐÊM:
- **Chạy thử siêu tốc (Smoke Test):** Đặt `DEBUG_MODE = True` ở Cell 2. Code sẽ chỉ nạp 40 mẫu/lớp và chạy 1 epoch trong ~30-45 giây.
- **Treo máy qua đêm:** Đổi thành `DEBUG_MODE = False`, sau đó nhấn **Save Version** -> **Save & Run All (Commit)** trên Kaggle.

In [ ]:
# ==============================================================================
# STEP 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN DỮ LIỆU
# ==============================================================================
!pip install -q transformers openpyxl emoji python-box scikit-learn

import os
import glob
import json
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
from box import Box

# Tự động dò tìm file dữ liệu train_conversations.json trên Kaggle hoặc Local
found_files = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
              glob.glob('./**/train_conversations.json', recursive=True) + \
              glob.glob('D:/KLTN/**/train_conversations.json', recursive=True) + \
              glob.glob('d:/Khóa Luận Tốt Nghiệp/**/train_conversations.json', recursive=True)

if not found_files:
    raise FileNotFoundError('❌ Không tìm thấy train_conversations.json! Hãy kiểm tra Input Dataset trên Kaggle.')

DATA_DIR = os.path.dirname(found_files[0])
print(f'--> ✅ Đã kết nối dữ liệu Stanceosaurus Reconstructed tại: {DATA_DIR}')


In [ ]:
# ==============================================================================
# STEP 2: CẤU HÌNH SIÊU THAM SỐ (ĐỒNG BỘ 100% ĐỂ SO SÁNH CÔNG BẰNG)
# ==============================================================================
DEBUG_MODE = False  # <-- ĐỔI THÀNH False KHI MUỐN TREO MÁY QUA ĐÊM TRÊN KAGGLE!

config = Box({
    'data_dir': DATA_DIR,
    'debug_mode': DEBUG_MODE,
    'debug_samples_per_class': 40,        # 40 mẫu/lớp khi chạy thử siêu tốc
    'model_name': 'vinai/bertweet-base',  # Đồng bộ chính xác backbone BERTweet
    'num_classes': 3,                     # 3: Supporting, Refuting, Other
    'max_seq_len': 128,
    'batch_size': 16,                     # Đồng bộ batch size = 16
    'accum_steps': 2,                     # Effective batch size = 32
    'learning_rate': 2e-5,                # Đồng bộ learning rate = 2e-5
    'weight_decay': 0.01,
    'num_epochs': 1 if DEBUG_MODE else 25,  # Trần epoch cao, dừng thật bằng Early Stopping (xem early_stop_patience)
    'early_stop_patience': 4,             # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 4 epoch liên tiếp
    'warmup_ratio': 0.1,
    'gru_hidden': 768,
    'gru_layer': 1,
    'alpha': 0.5,                         # Trọng số Target Prototypical Contrastive Loss (TPCL)
    'tau': 0.1,                           # Temperature cho Contrastive Loss
    'att_temp': 1.0,                      # Temperature cho Attention tương tác
    'loss_type': 'focal',                 # Đồng bộ Class-Balanced Focal Loss với baseline
    'focal_gamma': 2.0,
    'use_amp': True,                      # Mixed Precision FP16
    'seed': 42,
    'output_dir': './',
    'best_model_path': './best_sitpcl_bertweet_stanceosaurus.pt'
})

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

if config.debug_mode:
    print('⚠️ CHẾ ĐỘ: [DEBUG MODE] - Chạy thử nhanh trong 30-45 giây!')
else:
    print('🔥 CHẾ ĐỘ: [FULL TRAINING] - Chạy toàn bộ dữ liệu qua đêm!')
print(f'--> 🚀 Thiết bị: {config.device} | Max Epochs: {config.num_epochs} (early stop patience={config.early_stop_patience}) | Model Backbone: {config.model_name}')


In [ ]:
# ==============================================================================
# STEP 3: NẠP DỮ LIỆU HỘI THOẠI JSON ĐA BÊN (MULTI-PARTY CONVERSATIONS)
# ==============================================================================
def load_dialogue_json(filename):
    file_path = os.path.join(config.data_dir, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        samples = json.load(f)
        
    dialogues = []
    for s in samples:
        dialogues.append({
            'target': str(s['target']).strip(),
            'sentences': s['sentences'],
            'speakers': s['speakers'],
            'label': int(s['label'])
        })
    return pd.DataFrame(dialogues)

print('[*] Đang nạp dữ liệu hội thoại JSON đa bên...')
train_df = load_dialogue_json('train_conversations.json')
dev_df = load_dialogue_json('dev_conversations.json')
test_df = load_dialogue_json('test_conversations.json')

if config.debug_mode:
    k = config.debug_samples_per_class
    train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), k), random_state=config.seed)).reset_index(drop=True)
    dev_df = dev_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    test_df = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    print(f'⚠️ [DEBUG MODE] Đã trích mẫu: Train={len(train_df)}, Dev={len(dev_df)}, Test={len(test_df)}')
else:
    print(f'--> [FULL DATA] Train : {len(train_df):,} hội thoại | {train_df["target"].nunique()} targets')
    print(f'--> [FULL DATA] Dev   : {len(dev_df):,} hội thoại | {dev_df["target"].nunique()} targets')
    print(f'--> [FULL DATA] Test  : {len(test_df):,} hội thoại | {test_df["target"].nunique()} targets')

label_names = ['Supporting', 'Refuting', 'Other']
dist_df = pd.DataFrame({
    'Train': train_df['label'].value_counts().sort_index(),
    'Dev': dev_df['label'].value_counts().sort_index(),
    'Test': test_df['label'].value_counts().sort_index()
}, index=range(len(label_names)))
dist_df.index = label_names
display(dist_df)


In [ ]:
# ==============================================================================
# STEP 4: PROMPT TOKENIZATION CHO BERTWEET & SITPCL DATALOADER
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)
mask_token = tokenizer.mask_token  # '<mask>' cho RoBERTa/BERTweet
mask_token_id = tokenizer.mask_token_id

print(f'--> Tokenizer mask token: "{mask_token}" (ID: {mask_token_id})')

class SITPCLStanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = df.to_dict('records')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sentences = item['sentences']
        speakers = item['speakers']
        target = item['target']
        
        utterances = []
        for s_idx, sen in enumerate(sentences):
            # Prompt chuẩn hóa của SITPCL (ACL 2025):
            # "user {spk} says: {sen} . Stance towards {target} is <mask >"
            prompt = f'user {speakers[s_idx]} says: {sen} . Stance towards {target} is {mask_token}'
            enc = self.tokenizer(
                prompt,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids = enc['input_ids'].squeeze(0)
            att_mask = enc['attention_mask'].squeeze(0)
            
            mask_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]
            mask_idx = mask_indices[0].item() if len(mask_indices) > 0 else (att_mask.sum().item() - 2)
            
            utterances.append({
                'input_ids': input_ids,
                'attention_mask': att_mask,
                'mask_idx': mask_idx
            })
            
        return {
            'utterances': utterances,
            'speakers': speakers,
            'target': target,
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

def sitpcl_collate_fn(batch):
    all_input_ids = []
    all_att_masks = []
    all_mask_indices = []
    dia_idx = []
    speakers_list = []
    targets_list = []
    labels_list = []

    curr_idx = 0
    for sample in batch:
        n_utts = len(sample['utterances'])
        dia_idx.append((curr_idx, curr_idx + n_utts))
        curr_idx += n_utts

        for utt in sample['utterances']:
            all_input_ids.append(utt['input_ids'])
            all_att_masks.append(utt['attention_mask'])
            all_mask_indices.append(utt['mask_idx'])

        speakers_list.append(sample['speakers'])
        targets_list.append(sample['target'])
        labels_list.append(sample['label'])

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_att_masks),
        'mask_indices': torch.tensor(all_mask_indices, dtype=torch.long),
        'dia_idx': dia_idx,
        'speakers': speakers_list,
        'targets': targets_list,
        'labels': torch.stack(labels_list)
    }

train_dataset = SITPCLStanceDataset(train_df, tokenizer, config.max_seq_len)
dev_dataset = SITPCLStanceDataset(dev_df, tokenizer, config.max_seq_len)
test_dataset = SITPCLStanceDataset(test_df, tokenizer, config.max_seq_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=sitpcl_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)

print('--> ✅ DataLoader SITPCL đã khởi tạo xong!')


In [ ]:
# ==============================================================================
# STEP 5: KIẾN TRÚC MÔ HÌNH SITPCL (ATTENTION, SSE & TARGET PROTOTYPICAL CL)
# ==============================================================================
class Attention(nn.Module):
    def __init__(self, hidden_dim, temperature=1.0):
        super().__init__()
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** -0.5
        self.temperature = temperature
        self.act = nn.Softmax(dim=-1)

    def forward(self, query, keys, values):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        q = self.query_proj(query)
        k = self.key_proj(keys)
        v = self.value_proj(values)
        attn_scores = (torch.matmul(q, k.transpose(-2, -1)) * self.scale) / self.temperature
        attn_weights = self.act(attn_scores)
        output = torch.matmul(attn_weights, v)
        return output.squeeze(0) if output.size(0) == 1 else output

class SpeakerStanceExtractor(nn.Module):
    """Mô hình hóa tương tác người nói (Intra-speaker & Inter-speaker dependencies)"""
    def __init__(self, hidden_dim=768, temperature=1.0):
        super().__init__()
        self.linear_intra = nn.Linear(hidden_dim * 2, hidden_dim)
        self.linear_inter = nn.Linear(hidden_dim, hidden_dim)
        self.attention_intra = Attention(hidden_dim, temperature=temperature)
        self.attention_inter = Attention(hidden_dim, temperature=temperature)

    def forward(self, utterances, speakers):
        V_lst = []
        last_speaker_idx = dict()
        for i in range(len(speakers)):
            spk = speakers[i]
            if spk not in last_speaker_idx:
                V_lst.append(utterances[i])
            else:
                prev_idx = last_speaker_idx[spk]
                vh_concat = torch.cat((V_lst[prev_idx], utterances[i]), dim=-1)
                q_intra = self.linear_intra(vh_concat)
                c = utterances[:i+1]
                v_intra = self.attention_intra(q_intra, c, c)

                q_inter = self.linear_inter(utterances[i])
                k = torch.stack([V_lst[j] for j in range(prev_idx, i)]) if i > prev_idx else utterances[i].unsqueeze(0)
                v_inter = self.attention_inter(q_inter, k, k) if len(k) > 0 else torch.zeros_like(q_inter)

                V_lst.append(v_intra + v_inter)
            last_speaker_idx[spk] = i
        return torch.stack(V_lst)

def compute_target_prototypes(H_final, targets):
    """Tính toán Prototype vector cho từng target duy nhất trong mini-batch"""
    tar_to_utts = defaultdict(list)
    for i, t in enumerate(targets):
        tar_to_utts[t].append(H_final[i][-1])
        
    unique_targets = list(tar_to_utts.keys())
    prototypes = []
    for t in unique_targets:
        p_t = torch.stack(tar_to_utts[t]).mean(dim=0)
        prototypes.append(p_t)
    prototypes = torch.stack(prototypes)
    return prototypes, unique_targets

def target_prototypical_loss(H_final, targets, tau=0.1):
    """Target-aware Prototypical Contrastive Loss (TPCL chuẩn từ ACL 2025)"""
    if len(H_final) <= 1:
        return torch.tensor(0.0, device=H_final[0].device)
    
    prototypes, unique_targets = compute_target_prototypes(H_final, targets)
    prototypes = F.normalize(prototypes, p=2, dim=-1)   # [K, D]
    embeddings = torch.stack([F.normalize(h[-1], p=2, dim=-1) for h in H_final]) # [B, D]
    
    tar_map = {t: idx for idx, t in enumerate(unique_targets)}
    target_labels = torch.tensor([tar_map[t] for t in targets], device=embeddings.device)
    
    sim_matrix = torch.matmul(embeddings, prototypes.T) / tau  # [B, K]
    loss = F.cross_entropy(sim_matrix, target_labels)
    return loss

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weights=None):
        super().__init__()
        self.gamma = gamma
        self.weights = weights

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Tính class weights đồng bộ với Baseline
counts = [sum(train_df['label'] == c) for c in range(config.num_classes)]
class_weights = torch.tensor([1.0 / (max(c, 1) ** 0.5) for c in counts], dtype=torch.float).to(config.device)
class_weights = class_weights / class_weights.sum()

class SITPCLBERTweetModel(nn.Module):
    def __init__(self, config, class_weights):
        super().__init__()
        self.config = config
        self.backbone = AutoModel.from_pretrained(config.model_name)
        hidden_size = self.backbone.config.hidden_size
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=config.gru_hidden, num_layers=config.gru_layer, batch_first=True)
        self.sse = SpeakerStanceExtractor(hidden_dim=config.gru_hidden, temperature=config.att_temp)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(config.gru_hidden, config.num_classes)
        )
        self.focal_criterion = FocalLoss(gamma=config.focal_gamma, weights=class_weights)

    def forward(self, input_ids, attention_mask, mask_indices, dia_idx, speakers, targets, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state  # [total_utts, seq_len, hidden_size]
        
        batch_indices = torch.arange(last_hidden.size(0), device=last_hidden.device)
        mask_reps = last_hidden[batch_indices, mask_indices, :]  # [total_utts, hidden_size]
        
        H_final = []
        dialogue_representations = []
        for d_id, (st, ed) in enumerate(dia_idx):
            d_reps = mask_reps[st:ed] # [num_utts_in_dia, hidden_size]
            o, _ = self.gru(d_reps.unsqueeze(0))
            o = o.squeeze(0)
            v = self.sse(o, speakers[d_id])
            H_final.append(v)
            dialogue_representations.append(v[-1]) # Lấy phát ngôn cuối cùng (phát ngôn đích)
            
        stance_features = torch.stack(dialogue_representations)
        logits = self.classifier(stance_features)
        
        loss = None
        if labels is not None:
            focal_loss = self.focal_criterion(logits, labels)
            cl_loss = target_prototypical_loss(H_final, targets, tau=self.config.tau)
            loss = focal_loss + self.config.alpha * cl_loss
            
        return loss, logits

model = SITPCLBERTweetModel(config, class_weights).to(config.device)
print('--> ✅ Đã khởi tạo thành công SITPCL trên Backbone BERTweet với đầy đủ SIE & TPCL!')


In [ ]:
# ==============================================================================
# STEP 6: THIẾT LẬP OPTIMIZER & SCHEDULER
# ==============================================================================
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = (len(train_loader) // config.accum_steps) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)

print(f'--> ✅ Tổng số bước huấn luyện SITPCL: {total_steps} (Warmup: {warmup_steps})')


In [ ]:
# ==============================================================================
# STEP 7: VÒNG LẶP HUẤN LUYỆN VÀ VALIDATION
# ==============================================================================
def evaluate_sitpcl(model, loader, target_names):
    model.eval()
    all_preds, all_labels = [], []
    cls_list = list(range(len(target_names)))
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            mask_indices = batch['mask_indices'].to(config.device)
            labels = batch['labels'].to(config.device)

            with torch.cuda.amp.autocast(enabled=config.use_amp):
                _, logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    mask_indices=mask_indices,
                    dia_idx=batch['dia_idx'],
                    speakers=batch['speakers'],
                    targets=batch['targets'],
                    labels=None # Khi evaluate không cần tính contrastive loss
                )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, labels=cls_list, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    per_class_f1 = f1_score(all_labels, all_preds, labels=cls_list, average=None, zero_division=0)
    report = classification_report(all_labels, all_preds, labels=cls_list, target_names=target_names, zero_division=0, output_dict=True)
    return macro_f1, acc, per_class_f1, report

best_dev_f1 = 0.0
patience_counter = 0
history = []
best_model_path = config.best_model_path

print('\n[*] BẮT ĐẦU TRAINING SITPCL (BERTWEET BACKBONE)...')
for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{config.num_epochs}')
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        mask_indices = batch['mask_indices'].to(config.device)
        labels = batch['labels'].to(config.device)

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            loss, _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                mask_indices=mask_indices,
                dia_idx=batch['dia_idx'],
                speakers=batch['speakers'],
                targets=batch['targets'],
                labels=labels
            )
            loss = loss / config.accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.accum_steps

        if (step + 1) % config.accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        loop.set_postfix(loss=total_loss / (step + 1))

    avg_train_loss = total_loss / len(train_loader)
    dev_f1, dev_acc, dev_per_f1, _ = evaluate_sitpcl(model, dev_loader, label_names)
    
    print(f'--> [Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Dev Macro-F1: {dev_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}%')
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'dev_macro_f1': dev_f1, 'dev_acc': dev_acc})
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'    🏆 [Lưu Checkpoint Tốt Nhất] Dev F1 đạt: {best_dev_f1*100:.2f}%')
    else:
        patience_counter += 1
        print(f'    ⏳ [Không cải thiện] {patience_counter}/{config.early_stop_patience} epoch liên tiếp (Best: {best_dev_f1*100:.2f}%)')
        if patience_counter >= config.early_stop_patience:
            print(f'    🛑 [EARLY STOPPING] Dừng tại epoch {epoch} vì Dev Macro-F1 không cải thiện sau {config.early_stop_patience} epoch liên tiếp.')
            break


In [ ]:
# ==============================================================================
# STEP 8: ĐÁNH GIÁ TRÊN TẬP TEST ZERO-SHOT & XUẤT BẢNG KẾT QUẢ ĐỐI SOÁT
# ==============================================================================
print('\n' + '='*75)
print('[*] ĐÁNH GIÁ SITPCL TRÊN TẬP TEST ZERO-SHOT VỚI CHECKPOINT TỐT NHẤT')
print('='*75)

best_model_path = getattr(config, 'best_model_path', './best_sitpcl_bertweet_stanceosaurus.pt')

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=config.device))
    print(f'--> [✓] Đã nạp thành công checkpoint tốt nhất từ: {best_model_path}')
else:
    print('--> [!] Không tìm thấy file checkpoint đã lưu, sử dụng trọng số hiện tại của model.')

test_f1, test_acc, test_per_f1, test_report_dict = evaluate_sitpcl(model, test_loader, label_names)

res_dict = {
    'Model': 'SITPCL (BERTweet Backbone)',
    'Mode': 'DEBUG' if config.debug_mode else 'FULL',
    'Num_Classes': config.num_classes,
    'Loss_Type': f'Focal + {config.alpha}*TPCL',
    'Accuracy': round(test_acc * 100, 2),
    'Macro_F1': round(test_f1 * 100, 2)
}

for idx, name in enumerate(label_names):
    if idx < len(test_per_f1):
        res_dict[f'F1_{name}'] = round(test_per_f1[idx] * 100, 2)

results_df = pd.DataFrame([res_dict])
results_df.to_csv('stanceosaurus_sitpcl_results.csv', index=False)

print('\n--- 📊 BẢNG KẾT QUẢ SITPCL (BERTWEET) ĐỂ ĐỐI SOÁT CÔNG BẰNG ---')
display(results_df)

report_df = pd.DataFrame(test_report_dict).T
print('\n--- Báo Cáo Phân Loại Chi Tiết Từng Nhãn ---')
display(report_df.round(3))

# Đọc kết quả của Baseline để so sánh trực tiếp nếu có sẵn
baseline_csv = 'stanceosaurus_bertweet_results.csv'
if os.path.exists(baseline_csv):
    print('\n' + '='*75)
    print('🔥 BẢNG ĐỐI SÁNH CÔNG BẰNG GIỮA CÁC ĐỐI THỦ TRÊN STANCEOSAURUS:')
    print('='*75)
    b_df = pd.read_csv(baseline_csv)
    compare_df = pd.concat([b_df, results_df], ignore_index=True)
    display(compare_df)

if config.debug_mode:
    print('\n🎉 [XÁC NHẬN] CHẾ ĐỘ DEBUG ĐÃ CHẠY THÀNH CÔNG!')
    print('👉 BẠN CHỈ CẦN QUAY LẠI CELL 2, ĐỔI `DEBUG_MODE = False`, RỒI COMMIT TREO QUA ĐÊM TRÊN KAGGLE!')
else:
    print('\n✅ [HOÀN TẤT] ĐÃ CHẠY XONG FULL DỮ LIỆU SITPCL TRÊN BERTWEET!')
